In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Управління продуктивністю: функція Qiskit від Q-CTRL Fire Opal
*Дивись [довідник API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Функції Qiskit — це експериментальна можливість, доступна лише користувачам IBM Quantum&reg; Premium Plan, Flex Plan і On-Prem (через API IBM Quantum Platform). Вони перебувають у статусі попереднього релізу та можуть змінюватись.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Огляд
Fire Opal Performance Management дозволяє будь-кому легко отримувати значущі результати від квантових комп'ютерів у масштабі, не будучи експертом з квантового заліза. Під час запуску схем із Fire Opal Performance Management автоматично застосовуються методи придушення помилок на основі ШІ, що дає змогу масштабувати більші задачі з більшою кількістю гейтів і кубітів. Такий підхід зменшує кількість вимірювань (shots), необхідних для отримання правильної відповіді, без жодних додаткових накладних витрат — що суттєво економить і час обчислень, і кошти.

Performance Management придушує помилки та підвищує ймовірність отримання правильної відповіді на зашумленому залізі. Іншими словами, він збільшує відношення сигнал/шум. На зображенні нижче показано, як підвищена точність завдяки Performance Management може зменшити потребу в додаткових вимірюваннях на прикладі 10-кубітного алгоритму Квантового перетворення Фур'є. Маючи лише 30 вимірювань, Q-CTRL досягає порогу впевненості 99%, тоді як налаштування за замовчуванням (`QiskitRuntime` Sampler, `optimization_level`=3 та `resilience_level`=1, `ibm_sherbrooke`) вимагають 170 000 вимірювань. Отримуючи правильну відповідь швидше, ти суттєво економиш обчислювальний час.

![Візуалізація покращеного часу виконання](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

Функцію Performance Management можна використовувати з будь-яким алгоритмом, і її легко замінити стандартними [примітивами Qiskit Runtime](/guides/primitives). За лаштунками кілька методів придушення помилок працюють разом, запобігаючи виникненню помилок під час виконання. Усі методи конвеєра Fire Opal попередньо налаштовані та не залежать від алгоритму — ти завжди отримуєш найкращу продуктивність «з коробки».

Щоб отримати доступ до Performance Management, [звернись до Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Опис
Fire Opal Performance Management має два варіанти виконання, схожих на примітиви Qiskit Runtime, тому ти можеш легко замінити їх Q-CTRL Sampler і Estimator. Загальний робочий процес використання функції Performance Management такий:
1. Визнач свою схему (і оператори у випадку Estimator).
2. Запусти схему.
3. Отримай результати.

Щоб зменшити апаратний шум, Fire Opal використовує низку методів придушення помилок на основі ШІ, зображених на малюнку нижче. У Fire Opal весь конвеєр повністю автоматизований і не потребує жодного налаштування.

Конвеєр Fire Opal усуває потребу в додаткових накладних витратах, таких як збільшений квантовий час виконання або додаткові фізичні кубіти. Зверни увагу, що класичний час обробки залишається чинником (орієнтовні значення наведено в розділі [Бенчмарки](#benchmarks), де «Загальний час» відображає як класичну, так і квантову обробку). На відміну від пом'якшення помилок, яке вимагає накладних витрат у формі вибірки, придушення помилок Fire Opal працює як на рівні гейтів, так і на рівні імпульсів, щоб усунути різні джерела шуму та запобігти виникненню помилок. Завдяки запобіганню помилкам усувається потреба в дорогій постобробці.

На зображенні нижче показані методи придушення помилок, автоматизовані Fire Opal Performance Management.

![Візуалізація конвеєра придушення помилок](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

Функція пропонує два примітиви — Sampler і Estimator, — а їхні вхідні та вихідні дані розширюють реалізовану специфікацію [примітивів Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Бенчмарки
[Опубліковані результати алгоритмічних бенчмарків](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) демонструють значне покращення продуктивності для різних алгоритмів, зокрема Бернштейна–Вазірані, квантового перетворення Фур'є, пошуку Гровера, квантового алгоритму апроксимаційної оптимізації та варіаційного квантового eigensolver. У решті цього розділу наведено докладнішу інформацію про типи алгоритмів, які можна запускати, а також про очікувану продуктивність і час виконання.

Наступні незалежні дослідження демонструють, як Performance Management від Q-CTRL дає змогу проводити алгоритмічні дослідження рекордного масштабу:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) — квантове навчання на ядрах до 50 кубітів
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) — квантова оцінка фази до 33 кубітів
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) — квантове завантаження даних до 21 кубіта

У таблиці нижче наведено приблизний орієнтир щодо точності та часу виконання за попередніми бенчмарками на `ibm_fez`. Продуктивність на інших пристроях може відрізнятися. Час використання розраховано виходячи з припущення про 10 000 вимірювань на схему. Зазначена «Кількість кубітів» — це не жорстке обмеження, а приблизні пороги, за яких можна очікувати стабільно високої точності рішення. Задачі більшого розміру також були успішно розв'язані, і тестування за цими межами заохочується.

| Приклад    | Кількість кубітів | Точність | Міра точності | Загальний час (с) | Час використання (с) | Примітив (Режим) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Відсоток успіху (частка запусків, у яких правильна відповідь є бітрядком із найбільшою кількістю підрахунків)     | 10    | 8         | Sampler |
| Квантове перетворення Фур'є | 30Q              | 100% | Відсоток успіху (частка запусків, у яких правильна відповідь є бітрядком із найбільшою кількістю підрахунків)      | 10    | 8        | Sampler |
| Квантова оцінка фази  | 30Q   | 99.9998%  | Точність знайденого кута: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Квантова симуляція: модель Ізінга (15 кроків) | 20Q  | 99.775%   |  $A$ (визначено нижче)  |  60 (на крок)  | 15 (на крок)   | Estimator |
| Квантова симуляція 2: молекулярна динаміка (20 часових точок) | 34Q  |  96.78%  |  $A_{mean}$ (визначено нижче)   | 10 (на часову точку)    | 6 (на часову точку)  | Estimator |

Визначення точності вимірювання значення математичного сподівання — метрика $A$ визначається так:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
де $ \epsilon^{ideal} $ = ідеальне значення математичного сподівання, $ \epsilon^{meas} $ = виміряне значення математичного сподівання, $\epsilon^{ideal}_{max} $ = ідеальне максимальне значення, а $\epsilon^{ideal}_{min}$ = ідеальне мінімальне значення. $A_{mean}$ — це просто середнє значення $A$ за кількома вимірюваннями.

Ця метрика використовується тому, що вона інваріантна до глобальних зсувів і масштабування діапазону досяжних значень. Іншими словами, незалежно від того, чи зміщуєш ти діапазон можливих значень математичного сподівання вгору чи вниз, або збільшуєш розкид, значення $A$ залишається стабільним.
## Початок роботи
Fire Opal Performance Management використовує Qiskit v`2.0.0`, яка є рекомендованою версією. Підтримувані версії: Qiskit >=v`2.0.0`.
Автентифікуйся за допомогою свого [API-ключа IBM Quantum Platform](http://quantum.cloud.ibm.com/) та вибери функцію Qiskit так. (Цей фрагмент коду припускає, що ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Запусти схему**

Запусти схему та за бажанням визнач бекенд і кількість вимірювань.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Ти можеш використовувати знайомі [API Qiskit Serverless](/guides/serverless), щоб перевірити статус свого робочого навантаження функції Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Отримай результат**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Результати мають той самий формат, що й [результат Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Примітив Sampler
### Приклад Sampler
Використовуй примітив Sampler з Fire Opal Performance Management для запуску схеми Бернштейна–Вазірані. Цей алгоритм, що використовується для знаходження прихованого рядка з виходів функції «чорного ящика», є поширеним алгоритмом для бенчмаркінгу, оскільки має єдину правильну відповідь.
**1. Створи схему**

Визнач правильну відповідь алгоритму — прихований бітрядок — і схему Бернштейна–Вазірані. Ти можеш регулювати ширину схеми, просто змінивши `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Запусти схему**

Запусти схему та за бажанням визнач бекенд і кількість вимірювань.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Перевір [статус](/guides/functions#check-job-status) свого робочого навантаження функції Qiskit або отримай [результати](/guides/functions#retrieve-results) так:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Побудуй графік найкращих бітрядків**

Побудуй графік бітрядка з найбільшою кількістю підрахунків, щоб перевірити, чи є прихований бітрядок модою.